# 🔤Tokenization

## Mục tiêu bài học
- Hiểu cách Large Language Models (LLM) chia nhỏ văn bản thành tokens
- So sánh sự khác biệt giữa tokenization tiếng Việt và tiếng Anh
- Tính toán chi phí sử dụng API dựa trên số lượng tokens
- Tối ưu hóa prompt để tiết kiệm chi phí

## 📚 Phần 1: Giới thiệu về Tokenization

### Tokenization là gì?
Tokenization là quá trình chia nhỏ văn bản thành các đơn vị nhỏ hơn gọi là **tokens**. Đây là bước đầu tiên mà LLM thực hiện khi xử lý văn bản.

### Tại sao cần tokenization?
- LLM không xử lý trực tiếp văn bản, mà xử lý các con số (tokens)
- Mỗi token được chuyển đổi thành một vector số để model có thể hiểu
- Chi phí API được tính dựa trên số lượng tokens (input + output)

### Một số quy tắc cơ bản:
- 1 token ≈ 4 ký tự tiếng Anh
- 1 token ≈ ¾ từ tiếng Anh
- 1 từ tiếng Việt có thể tốn nhiều token hơn tiếng Anh (2-3 tokens)
- Khoảng trắng, dấu câu cũng tốn tokens

## 🛠️ Phần 2: Cài đặt và Chuẩn bị

Chúng ta sẽ sử dụng thư viện `tiktoken` - công cụ tokenization của OpenAI

In [1]:
# Cài đặt thư viện cần thiết
%pip install --upgrade tiktoken pandas
%pip install --upgrade torch --index-url https://download.pytorch.org/whl/cpu
%pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.4 MB 4.1 MB/s eta 0:00:03
   --- ------------------------------------ 0.8/10.4 MB 4.9 MB/s eta 0:00:02
   --------- ------------------------------ 2.5/10.4 MB 12.3 MB/s eta 0:00:01
   ----------- ---------------------------- 2.9/10.4 MB 12.2 MB/s eta 0:00:01
   ------------- -------------------------- 3.6/10.4 MB 11.4 MB/s eta 0:00:01
   ------------------- -------------------- 5.1/10.4 MB 14.3 MB/s eta 0:00:01
   ----------------------- ---------------- 6.1/10.4 MB 15.0 MB/s eta 0:00:01
   ------------------------------ --------- 7.9/10.4 MB 17.4 MB/s eta 0:00:01
   ------------------------------------- -- 9.8/10.4 MB 19.6 MB/s eta 0:00:01
   ---------------------------------------  10.4/10.4 MB 25.1 MB/s eta 0:00:01
   --------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import thư viện
import tiktoken
import pandas as pd
from typing import List

# Import transformers với xử lý lỗi
try:
    from transformers import AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except Exception as e:
    print(f"⚠️ Không thể import transformers: {e}")
    print("📝 Hàm so sánh BPE vs WordPiece sẽ chỉ hiển thị BPE")
    TRANSFORMERS_AVAILABLE = False

## 🔬 Phần 3: Thử nghiệm Tokenization

### 3.1 Khởi tạo Tokenizer
Chúng ta sẽ sử dụng tokenizer của GPT-4 (encoding: `cl100k_base`)

In [5]:
# Khởi tạo tokenizer cho GPT-4/GPT-3.5-turbo
encoding = tiktoken.get_encoding("cl100k_base")

# Hoặc có thể khởi tạo theo tên model cụ thể
# encoding = tiktoken.encoding_for_model("gpt-4")

print("✅ Tokenizer đã được khởi tạo thành công!")

✅ Tokenizer đã được khởi tạo thành công!


### 3.2 Hàm tiện ích để phân tích tokens

In [6]:
def analyze_text(text: str, encoding) -> dict:
    """Phân tích văn bản và trả về thông tin về tokens"""
    tokens = encoding.encode(text)
    
    return {
        'text': text,
        'num_tokens': len(tokens),
        'num_characters': len(text),
        'chars_per_token': round(len(text) / len(tokens), 2) if len(tokens) > 0 else 0
    }

def display_tokens(text: str, encoding):
    """Hiển thị chi tiết tokens của văn bản và trả về mảng tokens"""
    result = analyze_text(text, encoding)
    tokens = encoding.encode(text)
    
    print(f"📝 Văn bản: {result['text']}")
    # print(f"   - Số ký tự: {result['num_characters']}")
    # print(f"   - Số tokens: {result['num_tokens']}")
    # print(f"   - TB: {result['chars_per_token']} ký tự/token")
    print(f"   - Mảng tokens: {tokens}")
    
    # Hiển thị chi tiết từng token
    print(f"\n🔍 Chi tiết tokens:")
    for i, token in enumerate(tokens):
        decoded = encoding.decode([token])
        print(f"   Token {i+1}: {token} → '{decoded}'")
    
    return tokens

### 3.3 Thử nghiệm với tiếng Anh

In [7]:
# Ví dụ tiếng Anh
english_text = "Hello, how are you today?"
print("TIẾNG ANH:")
result_en = display_tokens(english_text, encoding)

TIẾNG ANH:
📝 Văn bản: Hello, how are you today?
   - Mảng tokens: [9906, 11, 1268, 527, 499, 3432, 30]

🔍 Chi tiết tokens:
   Token 1: 9906 → 'Hello'
   Token 2: 11 → ','
   Token 3: 1268 → ' how'
   Token 4: 527 → ' are'
   Token 5: 499 → ' you'
   Token 6: 3432 → ' today'
   Token 7: 30 → '?'


### 3.4 Thử nghiệm với tiếng Việt

In [8]:
# Ví dụ tiếng Việt
vietnamese_text = "Xin chào, bạn khỏe không?"
print("\nTIẾNG VIỆT:")
result_vi = display_tokens(vietnamese_text, encoding)


TIẾNG VIỆT:
📝 Văn bản: Xin chào, bạn khỏe không?
   - Mảng tokens: [55, 258, 523, 6496, 78, 11, 90537, 24040, 86242, 68, 54137, 30]

🔍 Chi tiết tokens:
   Token 1: 55 → 'X'
   Token 2: 258 → 'in'
   Token 3: 523 → ' ch'
   Token 4: 6496 → 'à'
   Token 5: 78 → 'o'
   Token 6: 11 → ','
   Token 7: 90537 → ' bạn'
   Token 8: 24040 → ' kh'
   Token 9: 86242 → 'ỏ'
   Token 10: 68 → 'e'
   Token 11: 54137 → ' không'
   Token 12: 30 → '?'


### 3.5 So sánh tiếng Anh vs tiếng Việt

In [8]:
# So sánh
print("\n" + "=" * 50)
print("SO SÁNH:")
print(f"Tiếng Anh: {result_en['num_tokens']} tokens")
print(f"Tiếng Việt: {result_vi['num_tokens']} tokens")
token_ratio = result_vi['num_tokens'] / result_en['num_tokens']
print(f"💡 Tiếng Việt tốn {token_ratio:.1f}x tokens!")


SO SÁNH:
Tiếng Anh: 7 tokens
Tiếng Việt: 12 tokens
💡 Tiếng Việt tốn 1.7x tokens!


### 3.6 So sánh thuật toán: BPE vs WordPiece

**BPE (Byte Pair Encoding):**
- Được sử dụng bởi GPT models (OpenAI)
- Chia nhỏ dựa trên các cặp byte xuất hiện nhiều nhất
- Tốt cho nhiều ngôn ngữ khác nhau

**WordPiece:**
- Được sử dụng bởi BERT models (Google)
- Chia nhỏ dựa trên xác suất tối đa hóa
- Thường sử dụng prefix `##` cho sub-word

In [11]:
def compare_tokenization_algorithms(text: str, bpe_encoding, wordpiece_model: str = 'vinai/phobert-base'):
    """
    So sánh cách chia token giữa BPE và WordPiece
    
    Args:
        text: Văn bản cần phân tích
        bpe_encoding: Tokenizer BPE (tiktoken)
        wordpiece_model: Tên model BERT để load tokenizer WordPiece
    
    Returns:
        dict chứa kết quả so sánh
    """
    # BPE tokenization (GPT)
    bpe_tokens = bpe_encoding.encode(text)
    bpe_decoded = [bpe_encoding.decode([token]) for token in bpe_tokens]
    
    # WordPiece tokenization (BERT)
    wp_tokens = []
    wp_decoded = []
    
    if TRANSFORMERS_AVAILABLE:
        try:
            wp_tokenizer = AutoTokenizer.from_pretrained(wordpiece_model)
            wp_tokens = wp_tokenizer.encode(text, add_special_tokens=False)
            wp_decoded = wp_tokenizer.convert_ids_to_tokens(wp_tokens)
        except Exception as e:
            print(f"⚠️ Lỗi khi load WordPiece tokenizer: {e}")
    else:
        print("⚠️ Transformers không khả dụng. Chỉ hiển thị kết quả BPE.")
    
    # Hiển thị kết quả
    print(f"📝 Văn bản gốc: '{text}'")
    print(f"\n{'='*60}")
    
    print(f"\n🔵 BPE (GPT-4) - {len(bpe_tokens)} tokens:")
    print(f"   Tokens: {bpe_tokens}")
    print(f"   Decoded: {bpe_decoded}")
    
    if wp_tokens:
        print(f"\n🟢 WordPiece (BERT) - {len(wp_tokens)} tokens:")
        print(f"   Tokens: {wp_tokens}")
        print(f"   Decoded: {wp_decoded}")
        
        print(f"\n{'='*60}")
        print(f"📊 So sánh:")
        print(f"   - BPE: {len(bpe_tokens)} tokens")
        print(f"   - WordPiece: {len(wp_tokens)} tokens")
        diff = len(bpe_tokens) - len(wp_tokens)
        print(f"   - Chênh lệch: {abs(diff)} tokens ({'BPE nhiều hơn' if diff > 0 else 'WordPiece nhiều hơn' if diff < 0 else 'Bằng nhau'})")
    
    return {
        'text': text,
        'bpe': {
            'tokens': bpe_tokens,
            'decoded': bpe_decoded,
            'count': len(bpe_tokens)
        },
        'wordpiece': {
            'tokens': wp_tokens,
            'decoded': wp_decoded,
            'count': len(wp_tokens)
        }
    }

In [7]:
# Ví dụ 1: Tiếng Anh đơn giản
print("VÍ DỤ 1: TIẾNG ANH ĐƠN GIẢN")
result1 = compare_tokenization_algorithms("Hello world!", encoding)

VÍ DỤ 1: TIẾNG ANH ĐƠN GIẢN
📝 Văn bản gốc: 'Hello world!'


🔵 BPE (GPT-4) - 3 tokens:
   Tokens: [9906, 1917, 0]
   Decoded: ['Hello', ' world', '!']

🟢 WordPiece (BERT) - 3 tokens:
   Tokens: [7592, 2088, 999]
   Decoded: ['hello', 'world', '!']

📊 So sánh:
   - BPE: 3 tokens
   - WordPiece: 3 tokens
   - Chênh lệch: 0 tokens (Bằng nhau)


In [8]:
# Ví dụ 2: Từ phức tạp
print("\n\nVÍ DỤ 2: TỪ PHỨC TẠP")
result2 = compare_tokenization_algorithms("unhappiness", encoding)



VÍ DỤ 2: TỪ PHỨC TẠP
📝 Văn bản gốc: 'unhappiness'


🔵 BPE (GPT-4) - 3 tokens:
   Tokens: [359, 71, 67391]
   Decoded: ['un', 'h', 'appiness']

🟢 WordPiece (BERT) - 4 tokens:
   Tokens: [4895, 3270, 9397, 9961]
   Decoded: ['un', '##ha', '##pp', '##iness']

📊 So sánh:
   - BPE: 3 tokens
   - WordPiece: 4 tokens
   - Chênh lệch: 1 tokens (WordPiece nhiều hơn)


In [13]:
# Ví dụ 3: Tiếng Việt
print("\n\nVÍ DỤ 3: TIẾNG VIỆT")
result3 = compare_tokenization_algorithms("Công ty này đang làm ăn phát đạt", encoding)



VÍ DỤ 3: TIẾNG VIỆT
📝 Văn bản gốc: 'Công ty này đang làm ăn phát đạt'


🔵 BPE (GPT-4) - 16 tokens:
   Tokens: [34, 24976, 13892, 97635, 15199, 526, 39015, 76, 220, 6845, 77, 1343, 17099, 15199, 20842, 83]
   Decoded: ['C', 'ông', ' ty', ' này', ' đ', 'ang', ' là', 'm', ' ', 'ă', 'n', ' ph', 'át', ' đ', 'ạ', 't']

🟢 WordPiece (BERT) - 8 tokens:
   Tokens: [3344, 6892, 23, 52, 47, 203, 1073, 208]
   Decoded: ['Công', 'ty', 'này', 'đang', 'làm', 'ăn', 'phát', 'đạt']

📊 So sánh:
   - BPE: 16 tokens
   - WordPiece: 8 tokens
   - Chênh lệch: 8 tokens (BPE nhiều hơn)


In [ ]:
# Ví dụ 4: Câu dài
print("\n\nVÍ DỤ 4: CÂU DÀI")
result4 = compare_tokenization_algorithms("The quick brown fox jumps over the lazy dog", encoding)

### 💡 Nhận xét về BPE vs WordPiece

**Ưu điểm BPE:**
- Linh hoạt với nhiều ngôn ngữ
- Xử lý tốt các từ hiếm
- Được sử dụng trong các model GPT hiện đại

**Ưu điểm WordPiece:**
- Tối ưu cho tiếng Anh
- Dễ phân tích với prefix `##`
- Hiệu quả cho các tác vụ NLU (Natural Language Understanding)

**Khi nào dùng gì:**
- Dùng BPE (GPT): Khi cần generate text, đa ngôn ngữ
- Dùng WordPiece (BERT): Khi cần phân tích, classification

## 💰 Phần 4: Tính toán chi phí API

### 4.1 Bảng giá các model phổ biến (tính đến 2026)

| Model | Input ($/1M tokens) | Output ($/1M tokens) |
|-------|---------------------|----------------------|
| GPT-4o | $2.50 | $10.00 |
| GPT-4o-mini | $0.15 | $0.60 |
| GPT-5 | $1.25 | $10.00 |
| Claude 4 Sonnet | $1.50 | $7.50 |
| Claude 3.5 Haiku | $0.80 | $4.00 |

In [ ]:
# Bảng giá ($/1M tokens) - Cập nhật 2026
PRICING = {
    'gpt-4o': {'input': 2.5, 'output': 10.0},
    'gpt-4o-mini': {'input': 0.15, 'output': 0.60},
    'gpt-5': {'input': 1.25, 'output': 10.0},
    'claude-4-sonnet': {'input': 1.5, 'output': 7.5},
    'claude-3.5-haiku': {'input': 0.8, 'output': 4.0},
}

def calculate_cost(input_tokens: int, output_tokens: int, model: str = 'gpt-4o') -> dict:
    """Tính chi phí sử dụng API"""
    pricing = PRICING[model]
    input_cost = (input_tokens / 1_000_000) * pricing['input']
    output_cost = (output_tokens / 1_000_000) * pricing['output']
    
    return {
        'model': model,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_cost': input_cost + output_cost
    }

def display_cost(cost_info: dict):
    """Hiển thị chi phí"""
    print(f"💰 {cost_info['model']}: ${cost_info['total_cost']:.6f}")

### 4.2 Ví dụ tính chi phí cho một cuộc hội thoại

In [5]:
# Tính chi phí cho một câu hỏi
user_prompt = "Giải thích về AI"
ai_response = "AI là trí tuệ nhân tạo..."

input_tokens = len(encoding.encode(user_prompt))
output_tokens = len(encoding.encode(ai_response))

print("VÍ DỤ TÍNH CHI PHÍ:")
for model in ['gpt-4o-mini', 'gpt-4o', 'claude-3.5-sonnet']:
    cost = calculate_cost(input_tokens, output_tokens, model)
    display_cost(cost)

VÍ DỤ TÍNH CHI PHÍ:
💰 gpt-4o-mini: $0.000008
💰 gpt-4o: $0.000130
💰 claude-3.5-sonnet: $0.000189


### So sánh 2 prompt

In [14]:
# Viết 2 prompt khác nhau
prompt_1 = "Hãy giải thích chi tiết về..."
prompt_2 = "Giải thích về..."

r1 = analyze_text(prompt_1, encoding)
r2 = analyze_text(prompt_2, encoding)

print(f"Prompt 1: {r1['num_tokens']} tokens")
print(f"Prompt 2: {r2['num_tokens']} tokens")
print(f"Tiết kiệm: {r1['num_tokens'] - r2['num_tokens']} tokens")

Prompt 1: 14 tokens
Prompt 2: 8 tokens
Tiết kiệm: 6 tokens


###  Tính chi phí dự án

In [15]:
# Tính chi phí cho chatbot
daily_users = 100
messages_per_user = 10
avg_input = 50
avg_output = 100

total_messages = daily_users * messages_per_user
cost_per_msg = calculate_cost(avg_input, avg_output, 'gpt-4o-mini')['total_cost']
daily_cost = cost_per_msg * total_messages

print(f"Chi phí/ngày: ${daily_cost:.2f}")
print(f"Chi phí/tháng: ${daily_cost * 30:.2f}")

Chi phí/ngày: $0.07
Chi phí/tháng: $2.03


## 📝 Tổng kết

**Những điều cần nhớ:**
1. Tiếng Việt tốn nhiều tokens hơn tiếng Anh (2-3x)
2. Chi phí = Input tokens + Output tokens
3. Cách tối ưu: Viết prompt ngắn gọn, chọn model phù hợp
4. Luôn tính toán chi phí trước khi triển khai

**Tài nguyên:**
- [OpenAI Tokenizer](https://platform.openai.com/tokenizer)
- [OpenAI Pricing](https://openai.com/pricing)